# Phase 2.3 — EDA et diagnostics d'équilibre

**Projet** : Évaluation d'impact GEFI — Delta du Saloum
**Notebook** : `python/01_eda.ipynb`

Ce notebook explore les données préparées par `00_prepare_data.py` et **diagnostique l'équilibre des groupes traitement/contrôle** sur les covariables pré-déterminées. C'est l'étape qui justifie l'approche méthodologique : si les groupes sont déjà bien équilibrés, l'entropy balancing aura peu d'impact ; s'ils sont déséquilibrés, le rebalancing devient indispensable.

## Setup

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
print('ROOT :', ROOT)
print('Fichiers :', [p.name for p in PROC.glob('*.parquet')])

In [ ]:
# Chargement du dataset analytique
df = pd.read_parquet(PROC / 'menages_analyse.parquet')
dico = pd.read_csv(PROC / 'dictionnaire_variables.csv')

print(f"Dataset : {df.shape[0]} ménages × {df.shape[1]} variables")
print(f"Variables documentées dans le dictionnaire : {len(dico)}")
df.head(3)

## 1. Composition de l'échantillon

In [ ]:
# Répartition par traitement et village
print('=== Répartition traitement/contrôle ===')
print(df['traitement'].value_counts(dropna=False).rename({1.0:'Traités', 0.0:'Contrôle'}).to_string())

print('\n=== Répartition par village ===')
village_tab = (df.groupby(['village','traitement'], dropna=False)
                 .size().reset_index(name='n')
                 .pivot_table(index='village', columns='traitement', values='n', fill_value=0))
village_tab.columns = ['Contrôle' if c==0.0 else 'Traités' for c in village_tab.columns]
village_tab['Total'] = village_tab.sum(axis=1)
print(village_tab.sort_values('Total', ascending=False).to_string())

In [ ]:
# Visualisation de la répartition
fig, ax = plt.subplots(figsize=(8, 4))
village_tab_no_total = village_tab.drop(columns='Total').sort_values(by=list(village_tab.drop(columns='Total').columns), ascending=True)
village_tab_no_total.plot(kind='barh', ax=ax, color=['#1f77b4', '#d62728'], width=0.7)
ax.set_xlabel("Nombre de ménages")
ax.set_ylabel('')
ax.set_title('Échantillon GEFI par village et statut de traitement')
ax.legend(title='Statut')
plt.tight_layout()
plt.show()

## 2. Qualité des données par groupe

Avant tout calcul, on regarde le taux de complétude des principales variables, séparément dans le traitement et le contrôle. Un déséquilibre fort de complétude entre les deux groupes est un signal d'attention.

In [ ]:
# Taux de complétude par variable et par groupe
analytical_vars = [c for c in df.columns if c not in ('hh_id', 'village', 'traitement', 'caseid')]

complet = (df.groupby('traitement')[analytical_vars]
             .apply(lambda d: d.notna().mean() * 100)
             .T)
complet.columns = ['Contrôle (% non-null)', 'Traités (% non-null)']
complet['Écart (pp)'] = (complet['Traités (% non-null)'] - complet['Contrôle (% non-null)']).round(1)
complet = complet.round(1)
print('Top 15 variables les plus complètes :')
print(complet.sort_values('Traités (% non-null)', ascending=False).head(15).to_string())
print('\nTop 10 variables les plus déséquilibrées en complétude :')
print(complet.reindex(complet['Écart (pp)'].abs().sort_values(ascending=False).index).head(10).to_string())

## 3. Distribution des covariables pré-déterminées

Vue d'ensemble des covariables qui entreront dans le modèle d'entropy balancing.

In [ ]:
# Covariables numériques continues
cov_num = ['taille_menage', 'age_chef', 'dependency_ratio']
cov_num_present = [c for c in cov_num if c in df.columns]

df_plot = df.copy()
df_plot['Groupe'] = df_plot['traitement'].map({0.0:'Contrôle', 1.0:'Traités'})

fig, axes = plt.subplots(1, len(cov_num_present), figsize=(4*len(cov_num_present), 4))
if len(cov_num_present) == 1: axes = [axes]
for ax, c in zip(axes, cov_num_present):
    sns.boxplot(data=df_plot.dropna(subset=[c, 'Groupe']),
                x='Groupe', y=c, ax=ax,
                order=['Contrôle', 'Traités'],
                palette=['#1f77b4', '#d62728'],
                hue='Groupe', legend=False)
    ax.set_title(c)
    ax.set_xlabel('')
plt.suptitle('Covariables continues par groupe', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Covariables catégorielles : tableaux de proportions
cov_cat = ['sexe_chef', 'matri_chef', 'ethnie_chef', 'educ_chef']
cov_cat_present = [c for c in cov_cat if c in df.columns]

for c in cov_cat_present:
    print(f'\n=== {c} ===')
    ct = pd.crosstab(df[c], df['traitement'], normalize='columns') * 100
    ct.columns = ['Contrôle %', 'Traités %']
    ct['Écart (pp)'] = (ct['Traités %'] - ct['Contrôle %']).round(1)
    print(ct.round(1).sort_values('Traités %', ascending=False).head(8).to_string())

## 4. Diagnostics d'équilibre traitement/contrôle

C'est la section centrale. On calcule pour chaque covariable pré-déterminée la **différence de moyenne standardisée** (*standardized mean difference*, SMD), métrique standard pour quantifier le déséquilibre entre groupes :

$$\text{SMD} = \frac{\bar{X}_T - \bar{X}_C}{\sqrt{(s_T^2 + s_C^2)/2}}$$

Conventions usuelles (Austin 2009, Stuart 2010) :
- $|\text{SMD}| < 0{,}10$ → équilibre acceptable
- $0{,}10 \le |\text{SMD}| < 0{,}25$ → déséquilibre modéré
- $|\text{SMD}| \ge 0{,}25$ → déséquilibre préoccupant — c'est typiquement là que l'entropy balancing devient nécessaire

In [ ]:
def smd_numeric(x_t, x_c):
    """Standardized mean difference pour variable continue ou binaire."""
    x_t = pd.Series(x_t).dropna()
    x_c = pd.Series(x_c).dropna()
    if len(x_t) < 2 or len(x_c) < 2:
        return np.nan
    s2 = (x_t.var(ddof=1) + x_c.var(ddof=1)) / 2
    if s2 <= 0:
        return np.nan
    return (x_t.mean() - x_c.mean()) / np.sqrt(s2)


def smd_categorical(series, trt):
    """Pour une variable catégorielle, retourne un SMD par modalité (binarisée)."""
    df_loc = pd.DataFrame({'v': series, 't': trt}).dropna()
    if df_loc.empty:
        return pd.Series(dtype=float)
    out = {}
    for mod in df_loc['v'].unique():
        bin_t = (df_loc.loc[df_loc['t']==1, 'v'] == mod).astype(float)
        bin_c = (df_loc.loc[df_loc['t']==0, 'v'] == mod).astype(float)
        out[mod] = smd_numeric(bin_t, bin_c)
    return pd.Series(out)

In [ ]:
# Calcul des SMD pour toutes les covariables
covariates_num = ['taille_menage', 'age_chef', 'dependency_ratio']
covariates_cat = ['sexe_chef', 'matri_chef', 'ethnie_chef', 'educ_chef']

smd_rows = []

for c in covariates_num:
    if c not in df.columns: continue
    s = smd_numeric(df.loc[df['traitement']==1, c],
                    df.loc[df['traitement']==0, c])
    smd_rows.append({'covariable': c, 'modalite': '(continue)', 'smd': s})

for c in covariates_cat:
    if c not in df.columns: continue
    smds = smd_categorical(df[c], df['traitement'])
    for mod, s in smds.items():
        smd_rows.append({'covariable': c, 'modalite': str(mod), 'smd': s})

smd_df = pd.DataFrame(smd_rows).dropna(subset=['smd'])
smd_df['abs_smd'] = smd_df['smd'].abs()
smd_df['variable'] = smd_df['covariable'] + ' : ' + smd_df['modalite']
smd_df = smd_df.sort_values('abs_smd', ascending=False)

print('SMD par covariable et modalité (trié par ampleur) :')
print(smd_df[['variable', 'smd', 'abs_smd']].round(3).to_string(index=False))

In [ ]:
# Love plot — visualisation iconique des diagnostics d'équilibre
to_plot = smd_df.sort_values('smd').copy()
colors = ['#d62728' if abs(s) >= 0.25 else
          ('#ff7f0e' if abs(s) >= 0.10 else '#2ca02c')
          for s in to_plot['smd']]

fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(to_plot))))
ax.scatter(to_plot['smd'], range(len(to_plot)), c=colors, s=60, zorder=3)
ax.axvline(0, color='black', lw=0.8)
ax.axvline(0.1, color='orange', lw=0.6, ls='--', alpha=0.7)
ax.axvline(-0.1, color='orange', lw=0.6, ls='--', alpha=0.7)
ax.axvline(0.25, color='red', lw=0.6, ls='--', alpha=0.7)
ax.axvline(-0.25, color='red', lw=0.6, ls='--', alpha=0.7)
ax.set_yticks(range(len(to_plot)))
ax.set_yticklabels(to_plot['variable'])
ax.set_xlabel('Différence de moyenne standardisée (SMD)')
ax.set_title("Love plot — déséquilibre traitement/contrôle avant pondération\n"
             "Vert : |SMD|<0.10 (OK) ; Orange : 0.10-0.25 (modéré) ; Rouge : ≥0.25 (préoccupant)")
ax.set_xlim(-max(to_plot['abs_smd'].max()*1.1, 0.35),
            max(to_plot['abs_smd'].max()*1.1, 0.35))
plt.tight_layout()
plt.savefig(ROOT / 'docs' / 'figures' / 'love_plot_avant.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture** : pour chaque covariable (et chaque modalité pour les catégorielles), le point représente l'écart standardisé entre traités et contrôle. Si tous les points sont dans la bande verte (|SMD| < 0.10), les groupes sont déjà comparables sur observables et l'entropy balancing aura peu d'effet. Si plusieurs points sont en orange ou rouge, le rebalancing devient indispensable.

## 5. Comparaisons brutes des outcomes (sans aucun contrôle)

Avant toute estimation rigoureuse, regardons à quoi ressemble une comparaison naïve traitement/contrôle pour chaque outcome. **À ne pas interpréter comme l'impact du projet** — c'est un point de départ qui sera corrigé en Phase 2.4.

In [ ]:
# Outcomes par catégorie de question
outcomes = {
    'Q1 — Autonomisation des femmes': [
        'n04_a_any', 'n04_b_any', 'n04_c_any', 'n04_d_any',
        'n11_femme', 'n13_femme', 'n15_femme', 'n17_femme_pouvoir_revenu',
        'score_autonomie_depenses'
    ],
    'Q2 — Revenus / pêche': [
        'revenu_total_2023', 'n_activites', 'pratique_peche',
        'e01', 'e03', 'e05', 'e11', 'e16', 'e18_1', 'e18_2'
    ],
    'Q3 — Soudure / dépenses / logement': [
        'mois_soudure_2023', 'depense_totale', 'n_categories_depense'
    ],
    'Q4 — Environnement': [
        'k01_1', 'k01_2', 'k01_3', 'k04', 'score_adaptation'
    ]
}

def naive_compare(out_var):
    if out_var not in df.columns:
        return None
    sub = df[['traitement', out_var]].dropna()
    if len(sub) < 10:
        return None
    g = sub.groupby('traitement')[out_var]
    means = g.mean()
    ns = g.size()
    diff = means.get(1, np.nan) - means.get(0, np.nan)
    # SE approximative pour la différence (variances inégales)
    se = np.sqrt(g.var(ddof=1).get(1, np.nan)/ns.get(1, 1) +
                 g.var(ddof=1).get(0, np.nan)/ns.get(0, 1))
    return {'mean_T': means.get(1, np.nan),
            'mean_C': means.get(0, np.nan),
            'diff': diff, 'se': se,
            'n_T': int(ns.get(1, 0)), 'n_C': int(ns.get(0, 0))}

for question, vars_ in outcomes.items():
    print(f'\n========== {question} ==========')
    rows = []
    for v in vars_:
        res = naive_compare(v)
        if res:
            rows.append({'outcome': v, **res})
    if rows:
        rdf = pd.DataFrame(rows)
        rdf['t-stat'] = (rdf['diff'] / rdf['se']).round(2)
        print(rdf.round(3).to_string(index=False))

In [ ]:
# Visualisation : les outcomes Q1 (autonomisation)
out_q1 = [v for v in outcomes['Q1 — Autonomisation des femmes'] if v in df.columns]
df_plot = df.copy()
df_plot['Groupe'] = df_plot['traitement'].map({0.0:'Contrôle', 1.0:'Traités'})

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, v in zip(axes.flat, out_q1):
    sub = df_plot[['Groupe', v]].dropna()
    if sub[v].nunique() <= 2:
        sns.barplot(data=sub, x='Groupe', y=v, ax=ax,
                    order=['Contrôle', 'Traités'],
                    palette=['#1f77b4', '#d62728'], errorbar='se',
                    hue='Groupe', legend=False)
    else:
        sns.boxplot(data=sub, x='Groupe', y=v, ax=ax,
                    order=['Contrôle', 'Traités'],
                    palette=['#1f77b4', '#d62728'],
                    hue='Groupe', legend=False)
    ax.set_title(v, fontsize=10)
    ax.set_xlabel('')
for ax in axes.flat[len(out_q1):]:
    ax.set_visible(False)
plt.suptitle("Q1 — Outcomes d'autonomisation (comparaison brute)", y=1.02)
plt.tight_layout()
plt.show()

## 6. Synthèse et orientations pour la Phase 2.4

À documenter ici en fonction de ce qu'on observe :

1. **Équilibre initial** — si les SMD sur les covariables sont majoritairement dans la zone verte (<0.10), l'entropy balancing aura un rôle de robustesse (les estimations brutes et ajustées convergeront). S'ils sont dans la zone orange/rouge, l'entropy balancing devient l'estimateur principal et la comparaison brute n'est pas interprétable.

2. **Qualité des données par groupe** — repérer les variables où la complétude diffère fortement entre traités et contrôle ; ce sont des variables où l'estimation effective porte sur une fraction du sous-échantillon et où les conclusions sont moins solides.

3. **Premiers signaux d'impact** — les outcomes dont la comparaison brute montre des écarts importants méritent attention en Phase 2.4 — mais on attend les estimations entropy-balancées avant toute conclusion.

4. **Limites confirmées** — les 5 grappes restent une contrainte forte sur l'inférence : même si les SMD sont propres après pondération, les intervalles de confiance sur le coefficient de traitement seront larges. À assumer dans le rapport final.